# parameter-wrap-around-tensor — ex1: observe the composition-Parameter anti-pattern breaks isinstance

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `parameter-wrap-around-tensor`. Running the final beacon cell reports progress against the `Backprop: Parameter wrap around Tensor` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Parameter wrap around Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`parameter-wrap-around-tensor`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "parameter-wrap-around-tensor"
DD_SUBTOPIC = "Backprop: Parameter wrap around Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Parameter as wrapper-around-Tensor — quick refresher (anti-pattern)

A tempting but WRONG `Parameter` design is to use **composition** — store the wrapped tensor as an attribute:

```python
# anti-pattern: NOT how nn.Parameter is built
class WrapParam:
    def __init__(self, tensor):
        self.tensor = tensor             # composition, not inheritance
```

Why this looks fine at first: `WrapParam(t.zeros(3)).tensor` is the raw tensor, every op you want to do still works on `.tensor` directly.

Why it's actually broken: **`isinstance(p, MiniTensor)` returns `False`**. Every helper in the autograd layer — `build_parents`, `unbox_args`, `get_children`, `parameters()` — filters by `isinstance(_, MiniTensor)`. A wrapped-Parameter is silently skipped by all of them, so:

- The optimizer never sees it.
- The reverse pass never accumulates grad on it.
- The compute graph treats it like a plain Python object — invisible.

The right design is inheritance: `class Parameter(MiniTensor): ...`. The drill below makes you reproduce the broken-by-composition version and *observe* the silent failure.

### Exercise 1 — observe the composition-Parameter anti-pattern breaks isinstance

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the composition-Parameter (HAS-A) design vs the subclass-Parameter (IS-A) design by building both, then asserting that only the IS-A form survives the `isinstance(_, MiniTensor)` filter that every autograd helper depends on.
> Keywords: parameter, composition, anti-pattern, isinstance, is-a-vs-has-a
> ```

**KCs targeted:** `parameter-wrap-around-tensor`, `parameter-subclass-of-tensor`

Implement TWO Parameter designs and a comparison helper, so the test cell can demonstrate the silent-failure mode of the wrong one.

**1. `WrapParam(tensor)`** — the COMPOSITION design (anti-pattern).
   ```python
   class WrapParam:
       def __init__(self, tensor):
           self.tensor = tensor   # stored as an attribute
           self.requires_grad = True
   ```
   Does NOT inherit from `MiniTensor`.

**2. `IsAParam(MiniTensor)`** — the SUBCLASS design (correct).
   ```python
   class IsAParam(MiniTensor):
       def __init__(self, array):
           super().__init__(array, requires_grad=True)
   ```

**3. `collect_params_via_isinstance(things)`** — the helper every autograd layer uses. Returns the subset of `things` that pass `isinstance(_, MiniTensor)`:
   ```python
   def collect_params_via_isinstance(things):
       return [x for x in things if isinstance(x, MiniTensor)]
   ```

The point of the drill: when `collect_params_via_isinstance` is given a mixed bag containing both a `WrapParam` and an `IsAParam`, the WrapParam is **silently dropped**. The test cell asserts this — you're observing the bug, not fixing it.

Why this matters: every helper in your autograd layer (`build_parents`, `unbox_args`, `parameters()` walker) uses exactly this isinstance gate. Composition-Parameters become invisible trainable state. Use IS-A.

In [ ]:
class WrapParam:
    """Composition (HAS-A) Parameter — the WRONG design."""
    def __init__(self, tensor):
        self.tensor = tensor
        self.requires_grad = True


class IsAParam(MiniTensor):
    """Subclass (IS-A) Parameter — the RIGHT design."""
    def __init__(self, array):
        super().__init__(array, requires_grad=True)


def collect_params_via_isinstance(things: list) -> list:
    return [x for x in things if isinstance(x, MiniTensor)]


<details><summary>Solution</summary>

```python
class WrapParam:
    """Composition (HAS-A) Parameter — the WRONG design."""
    def __init__(self, tensor):
        self.tensor = tensor
        self.requires_grad = True


class IsAParam(MiniTensor):
    """Subclass (IS-A) Parameter — the RIGHT design."""
    def __init__(self, array):
        super().__init__(array, requires_grad=True)


def collect_params_via_isinstance(things: list) -> list:
    return [x for x in things if isinstance(x, MiniTensor)]
```

**The composition-vs-inheritance choice is load-bearing.** It's tempting to start with `WrapParam(tensor)` because composition 'feels safer' — no inheritance, no MRO surprises. But the entire autograd scaffolding consists of helpers that say `if isinstance(x, MiniTensor): ...`. A composition-Parameter fails that gate; it becomes invisible to `build_parents`, `unbox_args`, `get_children`, and `parameters()`.

**Why this fails silently.** No exception is raised. No warning. The training loop runs, the loss decreases (because *some* parameters are still updating), and a fraction of the model's weights stay frozen at their init values. Debugging this means inspecting `list(model.parameters())` and noticing it's mysteriously short.

**PyTorch's actual design.** `torch.nn.Parameter(torch.Tensor)` — subclassing, exactly the `IsAParam` form. The minimal class body in PyTorch consists of a `__new__` override (to handle the `requires_grad=True` default) and a `__deepcopy__` for state-dict copying. Nothing else. The IS-A relationship IS the design.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()